## Function Calling
- Function Calling(함수 호출)은 모델이 실시간 데이터에 접근하거나 특정 동작(API 호출, DB 수정 등)을 수행할 수 있게 해주는 가교 역할을 합니다.
쉽게 말해, 모델이 스스로 해결할 수 없는 일(예: "현재 서울 날씨 알려줘", "내 계좌 잔액 조회해줘")을 만났을 때, "아, 이 기능을 써서 나 대신 정보를 가져와 줘"라고 요청하는 기술입니다.

🛠️ Function Calling 작동 원리
모델이 직접 코드를 실행하는 것이 아니라, 어떤 함수를 어떤 인자(Parameter)로 호출해야 할지 결정해 주는 것이 핵심입니다.
- 함수 정의: 개발자가 사용할 함수와 그에 대한 설명을 모델에게 미리 알려줍니다.
- 질의: 사용자가 "내일 제주도 비 오니?"라고 묻습니다.
- 판단: 모델은 학습된 데이터에 내일 날씨가 없음을 인지하고, 등록된 get_weather(location, date) 함수가 적합하다고 판단합니다.
- 출력 (Tool Call): 모델은 텍스트 답변 대신 get_weather(location='제주', date='2026-04-14')라는 호출 정보를 내뱉습니다.
- 실행: 개발자의 코드(애플리케이션)가 실제로 날씨 API를 호출해 결과를 얻습니다.
- 최종 답변: API 결과를 다시 모델에게 전달하면, 모델이 이를 바탕으로 자연스러운 답변을 생성합니다.

In [1]:
# 함수 선언 
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project="byounghwa-go", location="global")

get_landmarks_function = {
    "name": "get_landmarks",
    "description": "주어진 지역의 랜드마크 정보를 가져옵니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "도시 이름 (예: 서울, 뉴욕)",
            },        
        },
        "required": ["location"],
    },
}

def get_landmarks(location) -> str:
    """Get information about landmarks.
 
    Args:
        location: 지역 이름
 
    Returns:
        랜드마크의 한 줄 요약
    """
   
    if location == "서울":
        # 예제를 위한 Dummy 데이터 반환
        return {"name": "경복궁", "summary": "조선 왕조 제일의 법궁"}
    else:
        return {"name": "정보 없음", "summary": "해당 위치의 랜드마크 정보가 없습니다."}

tools = types.Tool(function_declarations=[get_landmarks_function])
config = types.GenerateContentConfig(tools=[tools])

In [2]:
# 모델의 판단 
contents = [
    types.Content(
        role="user", parts=[types.Part(text="서울의 랜드마크 알려줘.")]
    )
]

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=contents,
    config=config,
)

# Gemini 응답에서 함수 호출 체크
tool_call = response.candidates[0].content.parts[0].function_call
if tool_call:
    print(f"Function: {tool_call.name}")
    print(f"Arguement: {tool_call.args}")
else:
    print("응답에서 함수가 호출되지 않았습니다.")
    print(response.text)

Function: get_landmarks
Arguement: {'location': '서울'}


In [3]:
#  함수 실행 
if tool_call.name == "get_landmarks":
    result = get_landmarks(**tool_call.args)
    print(f"Result: {result}")

# 결과 전달 및 최종 답변 생성 == #
    # 함수 응답 부분 직접 만들기
    function_response_part = types.Part.from_function_response(
        name=tool_call.name,
        response={"Result": result},
    )


    # 함수 호출 및 함수 실행 결과를 콘텐츠에 직접 추가
    contents.append(response.candidates[0].content)
    contents.append(types.Content(role="user", parts=[function_response_part]))


    final_response = client.models.generate_content(
        model="gemini-3-flash-preview",
        config=config,
        contents=contents,
    )

    print(final_response.text)

Result: {'name': '경복궁', 'summary': '조선 왕조 제일의 법궁'}
서울의 대표적인 랜드마크로는 **경복궁**이 있습니다. 경복궁은 조선 왕조 제일의 법궁으로, 한국의 역사와 전통을 느낄 수 있는 대표적인 장소입니다. 

그 외에도 다음과 같은 랜드마크들이 유명합니다.

*   **N서울타워 (남산타워):** 서울 시내를 한눈에 내려다볼 수 있는 전망대가 유명합니다.
*   **롯데월드타워:** 한국에서 가장 높은 건물로, 서울의 현대적인 스카이라인을 상징합니다.
*   **명동:** 쇼핑과 먹거리의 중심지로, 많은 관광객이 찾는 활기찬 거리입니다.
*   **DDP (동대문디자인플라자):** 독특한 건축 디자인으로 유명한 복합 문화 공간입니다.
*   **북촌 한옥마을:** 도심 속에서 전통 한옥의 정취를 느낄 수 있는 곳입니다.


In [4]:
# 자동 함수 호출(Only Python) 
def get_landmarks(location: str) -> dict:
    """
    사용자 질문에서 지역 이름을 추출하고 해당 지역의 랜드마크 정보를 반환합니다.


    Args:
        location: 지역 이름


    Returns:
        랜드마크 이름 및 한 줄 요약
    """
    if location == "서울":
        # 예제를 위한 Dummy 데이터 반환
        return {"name": "경복궁", "summary": "조선 왕조 제일의 법궁"}
    else:
        return {"name": "정보 없음", "summary": "해당 지역의 랜드마크 정보가 없습니다."}

config = types.GenerateContentConfig(tools=[get_landmarks])

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="서울의 랜드마크 알려줘.",
    config=config,
)

print(response.text)

서울의 대표적인 랜드마크로는 **경복궁**이 있습니다. 경복궁은 조선 왕조 제일의 법궁으로 알려져 있습니다. 그 외에도 남산서울타워, 롯데월드타워 등이 유명합니다. 더 궁금한 지역이 있으신가요?


In [5]:
# 병렬 함수 호출 

# 실제 함수 구현
def brew_coffee(strength: str) -> dict:
    """
    지정된 농도로 커피를 내립니다.


    Args:
        strength: 커피 농도 ('진하게' 또는 '부드럽게')


    Returns:
        status: 현재 커피 상태를 나타냅니다.
    """
    return {"status": f"{strength} 커피를 내리고 있습니다."}


def get_daily_briefing(topics: list[str] = ["news", "weather"]) -> dict:
    """
    지정된 주제에 대한 일일 브리핑을 가져옵니다.


    Args:
        topics: 브리핑에 포함할 주제 목록


    Returns:
        briefing_topics: 브리핑 주제
    """
    return {"briefing_topics": topics}


def set_thermostat(temperature_celsius: float = 22.0) -> dict:
    """
    실내 온도를 조절합니다.


    Args:
        temperature_celsius: 목표 온도 (섭씨)


    Returns:
        temperature: 온도 설정값.
    """
    return {"temperature": f"{temperature_celsius}°C"}

config = types.GenerateContentConfig(
    system_instruction="사용자의 모닝 루틴을 위해 필요한 모든 도구를 호출하세요.",
    tools=[brew_coffee, get_daily_briefing, set_thermostat]
)

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="좋은 아침! 하루를 시작할 준비를 해줘. 커피는 부드럽게~",
    config=config,
)

print(response.text)

좋은 아침입니다! 요청하신 대로 부드러운 커피를 준비하고 있으며, 오늘의 뉴스 정보와 날씨를 포함한 일일 브리핑을 가져왔습니다. 기분 좋은 하루 시작하시길 바랍니다!


In [6]:
# Function Calling 모드 

# Function Calling 모드 설정
tool_config = types.ToolConfig(
    function_calling_config=types.FunctionCallingConfig(
        mode="ANY",
        allowed_function_names=["<함수 이름>"] # NONE일 땐 allowed_function_names 필드는 설정이 불가합니다.
    )
)

# 생성 구성 생성
config = types.GenerateContentConfig(
    tools=[tools],
    tool_config=tool_config,
)

# 함수 호출 결과 확인
for part in response.candidates[0].content.parts:
    if part.function_call:
        fc = part.function_call
        print(f"함수: {fc.name}, 인자: {fc.args}")

config = types.GenerateContentConfig(
    system_instruction="사용자의 모닝 루틴을 위해 필요한 모든 도구를 호출하세요.",
    tools=[brew_coffee, get_daily_briefing, set_thermostat],
    tool_config = types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(
            mode="ANY",
            allowed_function_names=["brew_coffee", "get_daily_briefing", "set_thermostat"]
        )
    )

)

# 1. 첫 번째 요청
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="좋은 아침! 하루를 시작할 준비를 해줘. 커피는 부드럽게~",
    config=config,
)

# 2. 함수 실행 및 결과 수집
function_map = {
    "brew_coffee": brew_coffee,
    "get_daily_briefing": get_daily_briefing,
    "set_thermostat": set_thermostat,
}

function_responses = []
for part in response.candidates[0].content.parts:
    if part.function_call:
        fc = part.function_call
        result = function_map[fc.name](**fc.args)
        function_responses.append(
            types.Part.from_function_response(name=fc.name, response=result)
        )


# 3. 함수 결과를 모델에 전달하여 최종 응답 받기
final_response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
        types.Content(role="user", parts=[types.Part(text="좋은 아침! 하루를 시작할 준비를 해줘. 커피는 부드럽게~")]),
        response.candidates[0].content,  # 모델의 함수 호출
        types.Content(role="user", parts=function_responses),  # 함수 실행 결과
    ],
    config=types.GenerateContentConfig(
        system_instruction="사용자의 모닝 루틴을 위해 필요한 도구들을 호출해야 합니다.",
        tools=[brew_coffee, get_daily_briefing, set_thermostat],
    ),
)

print(final_response.text)  # 이제 텍스트 응답이 나옵니다

좋은 아침입니다! 활기찬 하루를 위해 다음 루틴을 준비했습니다.

1. **커피**: 요청하신 대로 **부드러운 농도**로 커피를 내리고 있습니다.
2. **실내 온도**: 쾌적한 아침을 위해 온도를 **22°C**로 맞췄습니다.
3. **일일 브리핑**: 오늘의 **주요 뉴스**와 **날씨 정보**가 준비되었습니다.

도움이 필요하시면 언제든 말씀해 주세요! 오늘도 좋은 하루 보내시길 바랍니다.
